# PCA as Optimal Linear Dimensionality Reduction for Embeddings — companion notebook

This notebook is the **narrative** half of the `pca-dimensionality-reduction` Python pillar. The
reusable, tested implementation lives next door in [`pca_dimensionality_reduction.py`](pca_dimensionality_reduction.py);
here we import it and walk the topic section by section, so every claim the page makes renders as
executed output.

**One source of truth.** `pca_dimensionality_reduction.py` owns the numbers — its `assert`-based
harness encodes PCA-two-ways, Eckart–Young, the projection-distortion identity, and the retrieval
recall comparison — and its `grid_table()` is mirrored *to the decimal* by `SpectrumLaboratory.tsx`.
This notebook never redefines that math; it calls into it.

Run from the repo root:

```
uv run --with numpy --with scipy --with scikit-learn --with jupyter \
    jupyter execute notebooks/pca-dimensionality-reduction/01_pca_dimensionality_reduction.ipynb
```


In [ ]:
import sys
import pathlib

# Make pca_dimensionality_reduction.py importable whether the kernel starts in the repo
# root or in this notebook's own directory.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "pca-dimensionality-reduction", pathlib.Path("notebooks/pca-dimensionality-reduction")):
    if (_cand / "pca_dimensionality_reduction.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
from pca_dimensionality_reduction import (
    K_GRID, structured_embeddings,
    pca_via_covariance, pca_via_svd, explained_variance_ratio, effective_rank,
    reconstruction_error_sq, random_projection_error_sq,
    projection_distortion, recall_after_projection,
    grid_table, finance_demo,
    test_pca_eig_svd_agree, test_rayleigh_first_pc,
    test_variance_reconstruction_equivalence, test_eckart_young,
    test_explained_variance_and_effective_rank, test_projection_distortion,
    test_recall_after_projection, test_sklearn_crosscheck, test_finance_spectrum,
)

## 1. Why reduce dimension, and PCA two ways

A retriever stores millions of $d$-dimensional embeddings ($d = 768, 1536, 3072$), and both memory and
nearest-neighbor latency scale with $d$. The previous topic showed real embeddings have low *effective*
rank, so most coordinates carry little variance. PCA finds the directions the cloud actually spreads
along: the top eigenvectors of the centered covariance $\Sigma = \tilde X^\top \tilde X/(n-1)$,
equivalently the right singular vectors of the centered data matrix $\tilde X$, with
$\lambda_i = s_i^2/(n-1)$. The two computations agree to machine precision, and the top eigenvector
maximizes the Rayleigh quotient $w^\top \Sigma w$.

In [ ]:
X, _ = structured_embeddings(600, 200, 20, n_clusters=3, seed=0)
w_cov, comps_cov, _ = pca_via_covariance(X)
w_svd, comps_svd, s, _ = pca_via_svd(X)
print("eigenvalues agree (eig vs SVD)?", np.allclose(w_cov, w_svd))
print("top 5 eigenvalues:", np.round(w_cov[:5], 4))

test_pca_eig_svd_agree()
test_rayleigh_first_pc()

## 2. Variance maximization is reconstruction-error minimization

For any orthonormal frame $W \in \mathbb{R}^{d\times k}$, the expected squared reconstruction error
splits by Pythagoras into total minus retained variance:

$$\mathbb{E}\lVert z - WW^\top z\rVert^2 = \operatorname{tr}(\Sigma) - \operatorname{tr}(W^\top \Sigma W).$$

So minimizing reconstruction error and maximizing retained variance are the same problem, and both are
solved by the top-$k$ eigenvectors, leaving residual $\sum_{i>k}\lambda_i$.

In [ ]:
test_variance_reconstruction_equivalence()
print("variance-maximization and reconstruction-error-minimization are the same problem.")

## 3. Eckart–Young–Mirsky

The best rank-$k$ approximation of $\tilde X$ is the truncated SVD $\tilde X_k$, with squared Frobenius
error exactly the tail of the spectrum,

$$\lVert \tilde X - \tilde X_k\rVert_F^2 = \sum_{i>k} s_i^2 = (n-1)\sum_{i>k}\lambda_i,$$

and no rank-$k$ matrix — in particular no random rank-$k$ projection — does better.

In [ ]:
X, _ = structured_embeddings(700, 150, 25, n_clusters=3, seed=3)
for k in (4, 16, 64):
    direct, tail = reconstruction_error_sq(X, k)
    rand = random_projection_error_sq(X, k, trials=8, seed=k)
    print(f"k={k:3}: Frobenius error^2 {direct:10.1f} = tail sum {tail:10.1f}  | random projection {rand:10.1f}")

test_eckart_young()

## 4. Explained variance and effective rank

The cumulative **explained-variance ratio** $\mathrm{EVR}(k) = \sum_{i\le k}\lambda_i / \sum_i \lambda_i$
says how much of the cloud's spread the top $k$ directions capture, and the **effective rank**
(participation ratio) $n_\text{eff} = (\sum\lambda_i)^2/\sum\lambda_i^2$ is a soft count of the
directions that matter — far below the ambient dimension for structured data, near it for isotropic.

In [ ]:
X, _ = structured_embeddings(800, 256, 24, n_clusters=3, decay=0.9, noise=0.04, seed=4)
ev, _, _ = pca_via_covariance(X)
evr = explained_variance_ratio(ev)
print(f"structured cloud in R^256: effective rank {effective_rank(ev):.1f}")
print(f"  EVR at k=2,8,32,128: {np.round([evr[1],evr[7],evr[31],evr[127]],4)}")
iso, _ = structured_embeddings(800, 256, 256, decay=1.0, noise=0.0, seed=5)
print(f"isotropic cloud in R^256: effective rank {effective_rank(pca_via_covariance(iso)[0]):.0f}")

test_explained_variance_and_effective_rank()

## 5. What survives projection: distortion = explained variance

The fraction of squared norm — and of mean pairwise squared distance — that the top-$k$ projection
keeps equals the explained-variance ratio, *exactly*, because both are read off the same spectrum:

$$\frac{\mathbb{E}\lVert P_k z\rVert^2}{\mathbb{E}\lVert z\rVert^2} = \mathrm{EVR}(k).$$

This is the bridge from "variance retained" to "retrieval geometry retained." Data-oblivious random
projection (Johnson–Lindenstrauss, the next topic) preserves distances for *any* data at the cost of
extra dimensions; PCA is data-dependent and optimal in expectation for *this* cloud.

In [ ]:
X, _ = structured_embeddings(900, 200, 20, n_clusters=3, seed=6)
for k in (4, 16, 64):
    rn, rp, evr_k = projection_distortion(X, k)
    print(f"k={k:3}: retained norm {rn:.5f}  retained pairwise distance {rp:.5f}  EVR {evr_k:.5f}")

test_projection_distortion()

## 6. Retrieval: recall after projection, PCA vs random

The honest test is whether nearest-neighbor retrieval survives the projection. Recall@10 of the full
nearest neighbors is monotone in the kept dimension and, because PCA keeps the directions the cloud
(and so the neighbor structure) lives in, it beats a data-oblivious random projection at every kept
dimension.

In [ ]:
X, _ = structured_embeddings(1000, 256, 24, n_clusters=3, seed=7)
queries, _ = structured_embeddings(120, 256, 24, n_clusters=3, seed=8)
kept = (8, 16, 32, 64, 128)
pca = recall_after_projection(X, queries, kept, method="pca")
rand = recall_after_projection(X, queries, kept, method="random", seed=9)
print(f"{'kept':>6}{'recall PCA':>12}{'recall random':>15}")
for k in kept:
    print(f"{k:>6}{pca[k]:>12.3f}{rand[k]:>15.3f}")

test_recall_after_projection()

## 7. The cross-check and the grid the viz mirrors

We confirm our SVD-based PCA against `sklearn.decomposition.PCA`, then print `grid_table()` — the scree,
reconstruction-error, recall, and scatter blocks that `SpectrumLaboratory.tsx` mirrors to the decimal.

In [ ]:
test_sklearn_crosscheck()
tbl = grid_table()
sc = tbl["scree"]
print(f"effective rank {sc['effective_rank']:.2f}; top eigenvalues "
      f"{', '.join(f'{v:.3f}' for v in sc['top_eigvals'][:6])} ...")
print(f"{'k':>6}{'EVR':>9}{'pca_err':>12}{'rand_err':>12}{'recall_pca':>12}{'recall_rand':>13}")
for e, r, k, ev in zip(tbl["error"], tbl["recall"], K_GRID, sc["evr_at_k"]):
    print(f"{k:>6}{ev:>9.4f}{e['pca_err']:>12.1f}{e['rand_err']:>12.1f}{r['pca']:>12.4f}{r['rand']:>13.4f}")

## 8. Finance case study, and the summary

Three topical clusters of $1536$-dimensional financial-document embeddings (a **synthetic** low-rank-
plus-noise stand-in, not a trained encoder): the cloud has low effective rank, keeping the top $128$
principal components retains nearly all the variance, and PCA retains far more retrieval recall than a
random projection of the same width. Every claim above is also an `assert` in
`pca_dimensionality_reduction.py`; running this notebook re-verifies them all.

In [ ]:
test_finance_spectrum()
print("\nAll claims verified — the page, the laboratory, and this notebook agree by construction.")